In [62]:
import pandas as pd
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
data = pd.read_csv('./data/YE_west.csv')

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
data.head()

,verse,label
0,2024,1
1,All you have to be is yourself,1
2,Believe in your flyness...conquer your shyness.,1
3,Burn that excel spread sheet,1
4,Decentralize,1


In [5]:
for x in data['verse']:
    print(x)

2024
All you have to be is yourself
Believe in your flyness...conquer your shyness.
Burn that excel spread sheet
Decentralize
Distraction is the enemy of vision
Everything you do in life stems from either fear or love
For me giving up is way harder than trying.
For me, money is not my definition of success. Inspiring people is a definition of success
Fur pillows are hard to actually sleep on
George Bush doesn't care about black people
Have you ever thought you were in love with someone but then realized you were just staring in a mirror for 20 minutes?
I care. I care about everything. Sometimes not giving a f#%k is caring the most.
I feel calm but energized
I feel like I'm too busy writing history to read it.
I feel like me and Taylor might still have sex
I give up drinking every week
I leave my emojis bart Simpson color
I love sleep; it's my favorite.
I make awesome decisions in bike stores!!!
I really love my Tesla. I'm in the future. Thank you Elon.
I still think I am the greatest.


In [6]:
# data[i for i in label i == 1 ]

In [7]:
def clean_string(txt):
    new_text = ''
    i = 0
    while i < len(txt):

        # stop when '(' is found
        if txt[i] == '(':
            break

        # skip http links
        if txt[i:i+4] == "http":
            # skip until a space or end
            while i < len(txt) and txt[i] != ' ':
                i += 1
            continue

        # skip punctuation
        if txt[i] in ['.', "'", '!','“','”', '@']:
            i += 1
            continue

        new_text += txt[i]
        i += 1

    return new_text


In [8]:
clean_string("'Happy'! boy.(kumar)")

'Happy boy'

In [9]:
clean_string("DEAR FUTURE, I STILL BELIEVE IN YOU PRINTED IN THE NEW YORK TIMES THIS MORNING https://t.co/3hGgcjHzRE https://t.co/7tlMR2wa0q")

'DEAR FUTURE, I STILL BELIEVE IN YOU PRINTED IN THE NEW YORK TIMES THIS MORNING  '

In [10]:
for x in range(len(data['verse'])):
    value = clean_string(data.loc[x, 'verse'])
    data.loc[x, 'verse'] = value


In [11]:
data.loc[390, 'verse']

'This in not hate We are love God is love '

In [12]:
y = data['label']

In [13]:
print(f'Count of Ye:{(len([i for i in y if i==1]))}')
print(f'Count of NOT Ye:{(len([i for i in y if i==0]))}')

Count of Ye:4073
Count of NOT Ye:300


In [14]:
X = data['verse']

In [15]:
vocab = {'<UNK>':0}

In [16]:
def build_vocab():
    string=''
    for x in data['verse']:
        string =string +' '+ x
    value = string.split()

    for x in value:
        if x not in vocab:
            vocab[x] = len(vocab)

In [17]:
build_vocab()

In [18]:
vocab

{'<UNK>': 0,
 '2024': 1,
 'All': 2,
 'you': 3,
 'have': 4,
 'to': 5,
 'be': 6,
 'is': 7,
 'yourself': 8,
 'Believe': 9,
 'in': 10,
 'your': 11,
 'flynessconquer': 12,
 'shyness': 13,
 'Burn': 14,
 'that': 15,
 'excel': 16,
 'spread': 17,
 'sheet': 18,
 'Decentralize': 19,
 'Distraction': 20,
 'the': 21,
 'enemy': 22,
 'of': 23,
 'vision': 24,
 'Everything': 25,
 'do': 26,
 'life': 27,
 'stems': 28,
 'from': 29,
 'either': 30,
 'fear': 31,
 'or': 32,
 'love': 33,
 'For': 34,
 'me': 35,
 'giving': 36,
 'up': 37,
 'way': 38,
 'harder': 39,
 'than': 40,
 'trying': 41,
 'me,': 42,
 'money': 43,
 'not': 44,
 'my': 45,
 'definition': 46,
 'success': 47,
 'Inspiring': 48,
 'people': 49,
 'a': 50,
 'Fur': 51,
 'pillows': 52,
 'are': 53,
 'hard': 54,
 'actually': 55,
 'sleep': 56,
 'on': 57,
 'George': 58,
 'Bush': 59,
 'doesnt': 60,
 'care': 61,
 'about': 62,
 'black': 63,
 'Have': 64,
 'ever': 65,
 'thought': 66,
 'were': 67,
 'with': 68,
 'someone': 69,
 'but': 70,
 'then': 71,
 'realized': 7

In [19]:
def text_to_indices(txt):
    txt = txt.split()

    if len(txt) == 0:
        return [0]  # UNK token so seq_len = 1, NOT zero

    int_val = []
    for x in txt:
        if x in vocab:
            int_val.append(vocab[x])
        else:
            int_val.append(0)

    return int_val


In [20]:
text_to_indices('I love music')

[79, 33, 464]

In [21]:
class CustomDataset(Dataset):
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            torch.tensor(text_to_indices(self.data[idx]), dtype=torch.long),
            torch.tensor(self.label[idx], dtype=torch.float32)
        )


In [22]:
X.head(1)

0    2024
Name: verse, dtype: object

In [23]:
dataCus = CustomDataset(X,y)
print(len(dataCus))
dataCus[10]

4373


(tensor([58, 59, 60, 61, 62, 63, 49]), tensor(1.))

In [24]:
datum = DataLoader(dataCus, batch_size=1,shuffle=True)

In [25]:
torch.is_tensor(x)

False

In [26]:
for x,y in datum:
    print(f'X : {x}{x.dtype}, Y : {y}{y.dtype}')

X : tensor([[2582,   68, 2830, 3639,   96,   21, 4854, 4855, 4856,   96,   21, 4857,
         4858, 4859, 1723]])torch.int64, Y : tensor([1.])torch.float32
X : tensor([[6507, 1959, 2080, 6508, 6509, 6510, 3699]])torch.int64, Y : tensor([1.])torch.float32
X : tensor([[ 729, 5397,  232, 1652, 1650, 5398, 1234, 1722,   68, 5399, 5400, 5401]])torch.int64, Y : tensor([1.])torch.float32
X : tensor([[ 346,    4,   21,  183,  161, 6105, 1709, 4866,   53, 5787,  141, 6106,
           40, 6107, 3521,   11,  307, 6108]])torch.int64, Y : tensor([1.])torch.float32
X : tensor([[672,   7,  50, 673,  44,  50, 674]])torch.int64, Y : tensor([0.])torch.float32
X : tensor([[1853,  940, 1854, 1855,  947, 1856, 1857, 1858, 1391,  849, 1854, 1859,
         1427, 1860, 1324, 1300, 1442, 1318, 1268]])torch.int64, Y : tensor([1.])torch.float32
X : tensor([[  79, 1470,   50, 7927,   23, 7928, 7929, 1592,   96,  111, 7930,  375,
           23,   45,  112, 4926,   23,  271, 2496]])torch.int64, Y : tensor([1.])torc

In [27]:
class ModelYe(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,64)
        self.rnn = nn.RNN(64,128,batch_first=True)
        self.linear = nn.Linear(128,1)

    def forward(self, data):
        output = self.embedding(data)
        hidden, output = self.rnn(output)
        # hidden is the hidden_state, Which is not   requred for the model
        output = self.linear(output)
        return(output)



In [28]:
model = ModelYe(len(vocab)).to(device)

In [35]:
learning_rate = 0.001
epochs = 5

In [30]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [63]:
# Traning on CPU
# for epoch in range(epochs):
#     total_loss = 0
#     for x, y in datum:
#         output = model(x)
#         output = output.squeeze(2).squeeze(1)
#         # print(f'Model-out Shape :{output.dtype}')
#         # print(f'Y shape :{y.dtype}')
#         loss = criterion(output,y)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     print(f'Epoch:{epoch + 1}/{epochs}, Loss:{total_loss} ')


In [36]:
for epoch in range(epochs):
    total_loss = 0
    for x, y in datum:
        model.train()
        x = x.to(device)
        output = model(x)
        output = output.squeeze(2).squeeze(1)
        # print(f'Model-out Shape :{output.dtype}')
        # print(f'Y shape :{y.dtype}')
        y = y.to(device)
        loss = criterion(output,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch:{epoch + 1}/{epochs}, Loss:{total_loss} ')


Epoch:1/5, Loss:0.5333705652573775 
Epoch:2/5, Loss:0.02207489608107807 
Epoch:3/5, Loss:0.00363677182833432 
Epoch:4/5, Loss:0.0009050586572136679 
Epoch:5/5, Loss:0.07645945788362374 


In [ ]:
def predict_Ye(text):
    text = clean_string(text)
    text_val = text_to_indices(text)
    input_tensor = torch.tensor(text_val, dtype=torch.long).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits)
        if probs <= 0.1 : 
            value = "U NO Ye -" + ' Confidences:' + str(probs.item())
        else:
            value = 'YE % - '+ str(probs.item()*100)

    return value


In [38]:
predict_Ye('loving god')

'YE % - 100.0'

In [39]:
predict_Ye('Dreams work only when you do')

'U NO Ye - Confidences:1.75777281619105e-10'

In [40]:
predict_Ye('Consistency is more important than short bursts of motivation.')

'U NO Ye - Confidences:3.0571064343204446e-13'

In [41]:
predict_Ye('Shoot for the stars, so if you fall you land on a cloud')

'YE % - 99.99933242797852'

In [42]:
predict_Ye('''Everything I'm not makes me everything I am''')

'YE % - 99.99998807907104'

In [43]:
predict_Ye('''I have not failed. I've just found 10,000 ways that won't work.''')

'U NO Ye - Confidences:2.4838534007365354e-11'

In [56]:
def predict_proba(text):
    text = clean_string(text)
    text_val = text_to_indices(text)

    input_tensor = torch.tensor(text_val, dtype=torch.long).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.sigmoid(logits).item()
    return prob


def predict_label(text, threshold=0.5):
    return 1 if predict_proba(text) >= threshold else 0


In [58]:
y_true = data['label'].tolist()


In [ ]:
true_labels = []
pred_labels = []
for i in range(len(X)):
    text = X.iloc[i]
    true_labels.append(int(y_true[i]))   
    pred_labels.append(predict_label(text))
accuracy  = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, zero_division=0)
recall    = recall_score(true_labels, pred_labels, zero_division=0)
f1        = f1_score(true_labels, pred_labels, zero_division=0)
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [53]:
model.state_dict()

OrderedDict([('embedding.weight',
              tensor([[-0.2706,  1.3412,  2.1705,  ..., -0.7253, -1.1373, -1.8842],
                      [-0.1446,  0.1009, -0.7966,  ...,  0.1460, -1.8868,  0.7490],
                      [ 0.1487, -0.4121, -0.3133,  ..., -0.5039,  0.2521,  2.4976],
                      ...,
                      [ 0.6290, -1.0132, -0.4395,  ...,  1.7206,  0.4496,  2.9300],
                      [ 0.3267,  0.0050,  0.6823,  ..., -0.1641,  1.4738,  1.1648],
                      [ 0.2020,  1.2449,  0.9453,  ...,  1.4822, -1.7572,  2.6817]],
                     device='cuda:0')),
             ('rnn.weight_ih_l0',
              tensor([[-4.1347e-01, -4.0365e-01, -3.7936e-01,  ...,  5.7514e-02,
                       -1.4499e-01, -3.6758e-02],
                      [-6.4173e-01,  2.0326e-01,  7.3440e-02,  ..., -1.7170e-01,
                       -8.6827e-02,  2.9780e-01],
                      [-1.9465e-01,  5.1820e-01,  4.2838e-01,  ..., -2.2074e-02,
                 

In [49]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'vocab': vocab
}

torch.save(checkpoint, "YeModel.pth")
print("Saved YeModel.pth with vocab + weights")


Saved YeModel.pth with vocab + weights


In [ ]:
checkpoint = torch.load(
    "YeModel.pth",
    map_location=device,
    weights_only=False   
)

vocab = checkpoint['vocab']

model = ModelYe(len(vocab))
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print("Model + vocab loaded successfully!")


Model + vocab loaded successfully!


In [52]:
predict_Ye("Do not pity the dead, Harry. Pity the living, and, above all those who live without love.")


'U NO Ye - Confidences:1.0525622151646985e-08'